# ClariCast Data Assessment Report — Slide-by-Slide Tables & Presentation Charts (Slides 5–11)

Generates clean, widescreen (`16:9` aspect ratio, `300 DPI`) PNG charts and copy-paste-ready Excel tables mapped directly to each slide of **`ClariCast_Data Assessment Report.pptx`**:
- **Slide 5 (`Slide5_Missing_Values.png`):** Column-wise missing records & `%` bar chart + cell completeness & zero-demand stats.
- **Slide 6 (`Slide6_Monthly_Demand_Trend.png`):** Monthly demand line chart across all chemicals (`Oct 2022 – Dec 2025`) with annual averages and peak/trough callouts.
- **Slide 7 (`Slide7_Data_Grain_Levels.png`):** Record & unique entity counts across `Part`, `Site`, `Part-Region`, `Bulk-Region`, `Part-Region-Segment`, and `Bulk-Region-Segment`.
- **Slide 8 (`Slide8_Master_Hierarchy_Mapping.png`):** One-to-one vs. one-to-many mapping breakdown for `Part -> Site / Segment / Region` and `Bulk -> Site / Region / Segment`.
- **Slide 9 (`Slide9_Region_Segment_Concentration.png`):** Side-by-side `Quantity Share %` vs. `Part Share %` charts by **Region** and by **Business Segment**.
- **Slide 10 (`Slide10_Bulk_Sufficiency_and_Inactive_Segment.png`):** Inactive `Bulk Value` & true `<3` record insufficiency (excluding multi-year `2023–2025` series with 1–2 records in 2025) + Inactive Parts by Business Segment.
- **Slide 11 (`Slide11_Outliers_and_BulkRegion_Freq.png`):** Outlier classification chart (with the **zero bar removed**), clear method definitions, and outlier frequency at the **`Bulk × Region`** level.

## Cell 1: Imports, Presentation Styling & Data Loader

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# ==========================================
# CONFIGURATION
# ==========================================
FILE_PATH = "CC_History_Actuals_till_2025.csv"   # <-- Update to your CSV filename
OUT_DIR = "ClariCast_PPT_Assets"
EXCEL_OUT = os.path.join(OUT_DIR, "ClariCast_Slide_Tables.xlsx")
os.makedirs(OUT_DIR, exist_ok=True)

DAYFIRST = True
YEARS = [2022, 2023, 2024, 2025]
MIN_OBS_IQR = 8

# Clean corporate palette matching ClariCast slides
NAVY = "#1B365D"
TEAL = "#2A9D8F"
BLUE = "#3B7EA1"
ORANGE = "#E76F51"
AMBER = "#F4A261"
SLATE = "#6C757D"
LIGHT_BG = "#F8F9FA"

plt.rcParams.update({
    "font.sans-serif": "DejaVu Sans",
    "axes.edgecolor": "#CCCCCC",
    "axes.linewidth": 0.8,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

EXPECTED_COLS = [
    "Date", "Part Name", "Part Description", "Part Site", "Bulk Value", "Quantity",
    "Business Segment Value", "Business Segment Description", "Region Id"
]

raw_df = pd.read_csv(FILE_PATH, dtype=str, keep_default_na=False)
raw_df.columns = raw_df.columns.str.strip()

# Preserve literal "NA" (North America); convert true blanks/null sentinels to np.nan
NULL_SENTINELS = {"", "N/A", "NULL", "NONE", "NAN", "#N/A"}
df_raw = raw_df[EXPECTED_COLS].copy()
for col in df_raw.columns:
    s = df_raw[col].astype(str).str.strip()
    df_raw[col] = s.mask(s.str.upper().isin(NULL_SENTINELS))

# Parse Date (ISO first, dayfirst fallback) & Quantity
iso_dt = pd.to_datetime(df_raw["Date"], format="ISO8601", errors="coerce")
fb_mask = iso_dt.isna() & df_raw["Date"].notna()
iso_dt.loc[fb_mask] = pd.to_datetime(df_raw.loc[fb_mask, "Date"], format="mixed", dayfirst=DAYFIRST, errors="coerce")
df_raw["Date"] = iso_dt
df_raw["Quantity"] = pd.to_numeric(df_raw["Quantity"].str.replace(",", "", regex=False), errors="coerce")
df_raw["Year"] = df_raw["Date"].dt.year.astype("Int64")
df_raw["YearMonth"] = df_raw["Date"].dt.to_period("M")

# Apply the 4 cleaning rules for post-cleaning demand/coverage slides (Slides 6, 9, 10)
df_clean = df_raw.copy()
df_clean["Quantity"] = df_clean["Quantity"].clip(lower=0)
df_clean = df_clean[
    df_clean["Business Segment Description"].notna()
    & (~df_clean["Part Name"].fillna("").str.startswith("9"))
    & df_clean["Bulk Value"].notna()
    & df_clean[["Date", "Quantity", "Part Name", "Region Id"]].notna().all(axis=1)
].copy()

print(f"Raw Rows: {len(df_raw):,} | Clean Rows: {len(df_clean):,} | Assets folder: '{OUT_DIR}/'")

## Cell 2: Slide 5 — Data Structure & Completeness (Missing Value Chart & Stats)

In [ ]:
# ==========================================
# SLIDE 5: DATA STRUCTURE & COMPLETENESS
# ==========================================
s5_rows = []
for col in EXPECTED_COLS:
    miss = int(df_raw[col].isna().sum())
    pop = len(df_raw) - miss
    s5_rows.append({
        "Column": col,
        "Missing Records": miss,
        "Populated Records": pop,
        "Missing %": round(miss / len(df_raw) * 100, 2),
    })
df_s5_missing = pd.DataFrame(s5_rows).sort_values(["Missing Records", "Column"], ascending=[False, True]).reset_index(drop=True)

total_cells = df_raw[EXPECTED_COLS].size
missing_cells = int(df_raw[EXPECTED_COLS].isna().sum().sum())
cell_completeness_pct = (1 - missing_cells / total_cells) * 100

# Zero-demand share among Bulk x Region series active in 2025
br_2025 = df_clean[df_clean["Year"] == 2025].groupby(["Bulk Value", "Region Id"])["Quantity"].sum()
active_br_2025 = br_2025[br_2025 > 0].index
rows_in_active_br = df_clean.set_index(["Bulk Value", "Region Id"]).loc[lambda d: d.index.isin(active_br_2025)]
zero_rec_pct = (rows_in_active_br["Quantity"] == 0).mean() * 100

print(f"Cell Completeness: {cell_completeness_pct:.2f}% ({missing_cells:,} missing cells / {100 - cell_completeness_pct:.2f}%)")
print(f"Zero-demand records inside 2025-active Bulk x Region series: {zero_rec_pct:.2f}%")
display(df_s5_missing)

# Plot: Horizontal bar chart of Missing Records & Missing %
plot_s5 = df_s5_missing.sort_values("Missing Records", ascending=True)
fig, ax = plt.subplots(figsize=(9, 4.8))
bar_colors = [ORANGE if v > 0 else "#D0D7DE" for v in plot_s5["Missing Records"]]
bars = ax.barh(plot_s5["Column"], plot_s5["Missing Records"], color=bar_colors, height=0.62)

ax.set_title("Missing Records by Column (Raw Dataset: 490,281 Rows)", fontsize=12.5, fontweight="bold", color=NAVY, pad=12)
ax.set_xlabel("Missing Record Count", fontsize=10, color="#333333")
ax.set_xlim(0, max(plot_s5["Missing Records"].max() * 1.25, 10))
ax.xaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
ax.grid(axis="x", linestyle="--", alpha=0.4)
ax.grid(axis="y", visible=False)
sns.despine(top=True, right=True)

for bar, (_, r) in zip(bars, plot_s5.iterrows()):
    w = bar.get_width()
    lbl = f" {int(r['Missing Records']):,} ({r['Missing %']:.2f}%)" if r["Missing Records"] > 0 else " 0 (0.00%)"
    ax.text(w, bar.get_y() + bar.get_height() / 2, lbl, va="center", ha="left", fontsize=9,
            fontweight="bold" if r["Missing Records"] > 0 else "normal",
            color=NAVY if r["Missing Records"] > 0 else SLATE)

plt.tight_layout()
s5_png = os.path.join(OUT_DIR, "Slide5_Missing_Values.png")
plt.savefig(s5_png, dpi=300)
plt.show()

## Cell 3: Slide 6 — Historical Coverage & Monthly Demand Trend (All Chemicals)

In [ ]:
# ==========================================
# SLIDE 6: HISTORICAL COVERAGE & DEMAND TREND
# ==========================================
monthly_chem = (
    df_clean.groupby("YearMonth", as_index=False)
    .agg(
        Total_Quantity=("Quantity", "sum"),
        Active_Parts=("Part Name", lambda s: s[df_clean.loc[s.index, "Quantity"] > 0].nunique()),
        Active_Bulks=("Bulk Value", lambda s: s[df_clean.loc[s.index, "Quantity"] > 0].nunique()),
        Records=("Quantity", "size"),
    )
    .sort_values("YearMonth")
    .reset_index(drop=True)
)
monthly_chem["Year"] = monthly_chem["YearMonth"].dt.year
monthly_chem["YM_Str"] = monthly_chem["YearMonth"].astype(str)

# Annual summary table to accompany the slide
df_s6_annual = (
    monthly_chem.groupby("Year", as_index=False)
    .agg(
        Months_Covered=("YearMonth", "count"),
        Total_Quantity=("Total_Quantity", "sum"),
        Avg_Monthly_Quantity=("Total_Quantity", "mean"),
        Peak_Month=("YM_Str", lambda s: s.loc[monthly_chem.loc[s.index, "Total_Quantity"].idxmax()]),
        Peak_Month_Qty=("Total_Quantity", "max"),
    )
)
display(df_s6_annual)

# Plot: Monthly Demand Line Chart for All Chemicals (Oct 2022 - Dec 2025)
fig, ax = plt.subplots(figsize=(13, 5.2))
x = np.arange(len(monthly_chem))
y = monthly_chem["Total_Quantity"].values

ax.plot(x, y, marker="o", markersize=5.5, linewidth=2.3, color=NAVY, label="Monthly Total Demand (All Chemicals)")
ax.fill_between(x, y, color=TEAL, alpha=0.12)

# Plot annual mean reference segments
yr_palette = {2022: SLATE, 2023: BLUE, 2024: TEAL, 2025: ORANGE}
for yr, grp in monthly_chem.groupby("Year"):
    idx_vals = grp.index.values
    yr_mean = grp["Total_Quantity"].mean()
    ax.hlines(yr_mean, idx_vals[0], idx_vals[-1], colors=yr_palette.get(yr, SLATE),
              linestyles="--", linewidth=1.5, label=f"{yr} Avg: {yr_mean:,.0f}")

# Annotate peak and lowest full-month points
max_i = int(np.argmax(y))
min_i = int(np.argmin(y))
ax.annotate(f"Peak: {monthly_chem.loc[max_i, 'YM_Str']}\n({y[max_i]:,.0f})",
            xy=(max_i, y[max_i]), xytext=(max_i, y[max_i] * 1.08),
            ha="center", fontsize=8.5, fontweight="bold", color=NAVY,
            arrowprops=dict(arrowstyle="->", color=NAVY, lw=1))
ax.annotate(f"Low: {monthly_chem.loc[min_i, 'YM_Str']}\n({y[min_i]:,.0f})",
            xy=(min_i, y[min_i]), xytext=(min_i + 1.2, y[min_i] * 1.15),
            ha="left", fontsize=8.5, fontweight="bold", color=ORANGE,
            arrowprops=dict(arrowstyle="->", color=ORANGE, lw=1))

ax.set_xticks(x)
ax.set_xticklabels(monthly_chem["YM_Str"], rotation=65, ha="right", fontsize=8.5)
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
ax.set_ylim(0, y.max() * 1.22)
ax.set_title("Monthly Demand Trend Across All Chemicals (Oct 2022 – Dec 2025)", fontsize=13, fontweight="bold", color=NAVY, pad=12)
ax.set_ylabel("Total Quantity", fontsize=10)
ax.legend(ncol=5, loc="upper left", frameon=True, fontsize=8.5)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.grid(axis="x", visible=False)
sns.despine(top=True, right=True)

plt.tight_layout()
s6_png = os.path.join(OUT_DIR, "Slide6_Monthly_Demand_Trend.png")
plt.savefig(s6_png, dpi=300)
plt.show()

## Cell 4: Slide 7 — Data Grain & Transaction Structure

In [ ]:
# ==========================================
# SLIDE 7: DATA GRAIN & TRANSACTION STRUCTURE
# ==========================================
grain_specs = [
    ("1. Part Level", ["Part Name"]),
    ("2. Part Site Level", ["Part Site"]),
    ("3. Part × Region (Forecast Grain)", ["Part Name", "Region Id"]),
    ("4. Bulk × Region (WMAPE Eval Grain)", ["Bulk Value", "Region Id"]),
    ("5. Part × Region × Segment", ["Part Name", "Region Id", "Business Segment Description"]),
    ("6. Bulk × Region × Segment", ["Bulk Value", "Region Id", "Business Segment Description"]),
    ("7. Part × Site × Region × Segment", ["Part Name", "Part Site", "Region Id", "Business Segment Description"]),
]

s7_rows = []
for label, cols in grain_specs:
    sub_raw = df_raw.dropna(subset=cols)
    uniq_raw = len(sub_raw.drop_duplicates(subset=cols))
    sub_clean = df_clean.dropna(subset=cols)
    uniq_clean = len(sub_clean.drop_duplicates(subset=cols))
    s7_rows.append({
        "Aggregation Level": label,
        "Unique Entities (Raw)": uniq_raw,
        "Unique Entities (Clean)": uniq_clean,
        "Avg Raw Rows per Entity": round(len(sub_raw) / uniq_raw, 1) if uniq_raw else 0,
        "Avg Clean Rows per Entity": round(len(sub_clean) / uniq_clean, 1) if uniq_clean else 0,
    })

df_s7_grain = pd.DataFrame(s7_rows)
display(df_s7_grain)

# Plot: Horizontal bar chart of Unique Entities across hierarchy levels
plot_s7 = df_s7_grain.iloc[::-1].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(11, 5.2))
colors_s7 = [SLATE, SLATE, BLUE, TEAL, NAVY, SLATE, SLATE][::-1]
bars = ax.barh(plot_s7["Aggregation Level"], plot_s7["Unique Entities (Clean)"], color=colors_s7, height=0.58)

ax.set_title("Entity Granularity Structure (From 490,281 Raw Rows to Forecast & WMAPE Grains)",
             fontsize=12, fontweight="bold", color=NAVY, pad=12)
ax.set_xlabel("Distinct Combinations (Post-Cleaning)", fontsize=10)
ax.set_xlim(0, plot_s7["Unique Entities (Clean)"].max() * 1.24)
ax.xaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
ax.grid(axis="x", linestyle="--", alpha=0.4)
ax.grid(axis="y", visible=False)
sns.despine(top=True, right=True)

for bar, (_, r) in zip(bars, plot_s7.iterrows()):
    w = bar.get_width()
    ax.text(w, bar.get_y() + bar.get_height() / 2,
            f" {int(r['Unique Entities (Clean)']):,}  (avg {r['Avg Clean Rows per Entity']:.1f} rows/entity)",
            va="center", ha="left", fontsize=9, fontweight="bold", color="#222222")

plt.tight_layout()
s7_png = os.path.join(OUT_DIR, "Slide7_Data_Grain_Levels.png")
plt.savefig(s7_png, dpi=300)
plt.show()

## Cell 5: Slide 8 — Master Data Integrity & Hierarchy Mapping (1-to-1 vs. 1-to-Many)

In [ ]:
# ==========================================
# SLIDE 8: MASTER DATA INTEGRITY & HIERARCHY MAPPING
# ==========================================
hier_specs = [
    ("Part Name -> Part Site", "Part Name", "Part Site"),
    ("Part Name -> Business Segment", "Part Name", "Business Segment Description"),
    ("Part Name -> Region Id", "Part Name", "Region Id"),
    ("Bulk Value -> Part Site", "Bulk Value", "Part Site"),
    ("Bulk Value -> Region Id", "Bulk Value", "Region Id"),
    ("Bulk Value -> Business Segment", "Bulk Value", "Business Segment Description"),
]

s8_rows = []
for label, src, tgt in hier_specs:
    sub = df_clean[[src, tgt]].dropna().drop_duplicates()
    counts = sub.groupby(src)[tgt].nunique()
    n_src = int(len(counts))
    n_1to1 = int((counts == 1).sum())
    n_1tom = int((counts > 1).sum())
    s8_rows.append({
        "Hierarchy Relationship": label,
        "Distinct Source Entities": n_src,
        "Strict 1-to-1 Count": n_1to1,
        "Strict 1-to-1 %": round(n_1to1 / n_src * 100, 2) if n_src else 0,
        "1-to-Many Count": n_1tom,
        "1-to-Many %": round(n_1tom / n_src * 100, 2) if n_src else 0,
        "Max Targets per Source": int(counts.max()) if n_src else 0,
    })

df_s8_mapping = pd.DataFrame(s8_rows)
display(df_s8_mapping)

# Plot: 100% Stacked Horizontal Bar Chart of Strict 1-to-1 vs 1-to-Many
plot_s8 = df_s8_mapping.iloc[::-1].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(11, 4.8))
y_pos = np.arange(len(plot_s8))

b1 = ax.barh(y_pos, plot_s8["Strict 1-to-1 %"], color=TEAL, height=0.56, label="Strict 1-to-1 Mapping (%)")
b2 = ax.barh(y_pos, plot_s8["1-to-Many %"], left=plot_s8["Strict 1-to-1 %"], color=AMBER, height=0.56, label="1-to-Many Mapping (%)")

ax.set_yticks(y_pos)
ax.set_yticklabels(plot_s8["Hierarchy Relationship"], fontsize=9.5, fontweight="bold")
ax.set_xlim(0, 100)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title("Master Data Hierarchy Integrity: 1-to-1 vs. 1-to-Many Relationships", fontsize=12, fontweight="bold", color=NAVY, pad=12)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False, fontsize=9.5)
sns.despine(top=True, right=True)

for idx_r, r in plot_s8.iterrows():
    if r["Strict 1-to-1 %"] >= 12:
        ax.text(r["Strict 1-to-1 %"] / 2, idx_r, f"{r['Strict 1-to-1 %']:.1f}%\n({r['Strict 1-to-1 Count']:,})",
                va="center", ha="center", color="white", fontsize=8.5, fontweight="bold")
    if r["1-to-Many %"] >= 12:
        ax.text(r["Strict 1-to-1 %"] + r["1-to-Many %"] / 2, idx_r, f"{r['1-to-Many %']:.1f}%\n({r['1-to-Many Count']:,})",
                va="center", ha="center", color="#222222", fontsize=8.5, fontweight="bold")

plt.tight_layout()
s8_png = os.path.join(OUT_DIR, "Slide8_Master_Hierarchy_Mapping.png")
plt.savefig(s8_png, dpi=300, bbox_inches="tight")
plt.show()

## Cell 6: Slide 9 — Demand Concentration by Region & Business Segment

In [ ]:
# ==========================================
# SLIDE 9: DEMAND CONCENTRATION BY REGION & SEGMENT
# ==========================================
tot_clean_qty = df_clean["Quantity"].sum()
tot_clean_parts = df_clean["Part Name"].nunique()

df_s9_region = (
    df_clean.groupby("Region Id", as_index=False)
    .agg(Total_Quantity=("Quantity", "sum"), Distinct_Parts=("Part Name", "nunique"), Records=("Quantity", "size"))
    .assign(
        Quantity_Share_Pct=lambda d: (d["Total_Quantity"] / tot_clean_qty * 100).round(2),
        Part_Share_Pct=lambda d: (d["Distinct_Parts"] / tot_clean_parts * 100).round(2),
    )
    .sort_values("Total_Quantity", ascending=False)
    .reset_index(drop=True)
)

df_s9_segment = (
    df_clean.groupby("Business Segment Description", as_index=False)
    .agg(Total_Quantity=("Quantity", "sum"), Distinct_Parts=("Part Name", "nunique"), Records=("Quantity", "size"))
    .assign(
        Quantity_Share_Pct=lambda d: (d["Total_Quantity"] / tot_clean_qty * 100).round(2),
        Part_Share_Pct=lambda d: (d["Distinct_Parts"] / tot_clean_parts * 100).round(2),
    )
    .sort_values("Total_Quantity", ascending=False)
    .reset_index(drop=True)
)

print("--- Slide 9A: Region Concentration ---")
display(df_s9_region)
print("--- Slide 9B: Business Segment Concentration ---")
display(df_s9_segment)

# Two-panel side-by-side grouped bar charts
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.4))

# Left Graph: Quantity Share % vs Part Share % by Region
xr = np.arange(len(df_s9_region))
w = 0.36
r_b1 = ax1.bar(xr - w / 2, df_s9_region["Quantity_Share_Pct"], width=w, color=NAVY, label="Quantity Share (%)")
r_b2 = ax1.bar(xr + w / 2, df_s9_region["Part_Share_Pct"], width=w, color=TEAL, label="Part Share (%)")
ax1.set_xticks(xr)
ax1.set_xticklabels(df_s9_region["Region Id"], fontsize=9.5, fontweight="bold")
ax1.yaxis.set_major_formatter(mtick.PercentFormatter())
ax1.set_ylim(0, max(df_s9_region[["Quantity_Share_Pct", "Part_Share_Pct"]].max()) * 1.2)
ax1.set_title("Quantity Share vs. Part Share by Region", fontsize=12, fontweight="bold", color=NAVY, pad=10)
ax1.bar_label(r_b1, fmt="%.1f%%", padding=3, fontsize=8.5, fontweight="bold")
ax1.bar_label(r_b2, fmt="%.1f%%", padding=3, fontsize=8.5)
ax1.legend(frameon=False, fontsize=9)
ax1.grid(axis="y", linestyle="--", alpha=0.4)
ax1.grid(axis="x", visible=False)

# Right Graph: Quantity Share % vs Part Share % by Business Segment
seg_rev = df_s9_segment.iloc[::-1].reset_index(drop=True)
ys = np.arange(len(seg_rev))
h = 0.38
s_b1 = ax2.barh(ys + h / 2, seg_rev["Quantity_Share_Pct"], height=h, color=NAVY, label="Quantity Share (%)")
s_b2 = ax2.barh(ys - h / 2, seg_rev["Part_Share_Pct"], height=h, color=TEAL, label="Part Share (%)")
ax2.set_yticks(ys)
ax2.set_yticklabels(seg_rev["Business Segment Description"], fontsize=9)
ax2.xaxis.set_major_formatter(mtick.PercentFormatter())
ax2.set_xlim(0, max(seg_rev[["Quantity_Share_Pct", "Part_Share_Pct"]].max()) * 1.22)
ax2.set_title("Quantity Share vs. Part Share by Business Segment", fontsize=12, fontweight="bold", color=NAVY, pad=10)
ax2.bar_label(s_b1, fmt="%.1f%%", padding=3, fontsize=8, fontweight="bold")
ax2.bar_label(s_b2, fmt="%.1f%%", padding=3, fontsize=8)
ax2.legend(loc="lower right", frameon=False, fontsize=9)
ax2.grid(axis="x", linestyle="--", alpha=0.4)
ax2.grid(axis="y", visible=False)

sns.despine(top=True, right=True)
plt.tight_layout()
s9_png = os.path.join(OUT_DIR, "Slide9_Region_Segment_Concentration.png")
plt.savefig(s9_png, dpi=300)
plt.show()

## Cell 7: Slide 10 — Portfolio Coverage & History Sufficiency
- **Bulk Value Sufficiency:** Distinguishes inactive Bulks (`no 2025 demand`), **true `<3` record insufficiency (`2025-only launches with 1–2 records`)**, and **multi-year series (`2023/2024 + 2025`)** which are **NOT flagged as insufficient** even if they have only 1–2 records in 2025.
- **Inactive Part by Segment:** Active vs. Inactive distinct parts across every Business Segment.

In [ ]:
# ==========================================
# SLIDE 10: PORTFOLIO COVERAGE & HISTORY SUFFICIENCY
# ==========================================
def classify_bulk_sufficiency(df, group_cols):
    pos_df = df[df["Quantity"] > 0]
    all_ent = df.groupby(group_cols, as_index=False).agg(Total_Records=("Quantity", "size"), Total_Qty=("Quantity", "sum"))
    yr_recs = (
        pos_df.pivot_table(index=group_cols, columns="Year", values="Quantity", aggfunc="size", fill_value=0)
        .reindex(columns=YEARS, fill_value=0)
        .reset_index()
    )
    m = all_ent.merge(yr_recs, on=group_cols, how="left").fillna({y: 0 for y in YEARS})
    for y in YEARS:
        m[y] = m[y].astype(int)

    pos_2025 = m[2025]
    pos_prior = m[2022] + m[2023] + m[2024]

    conds = [
        (pos_2025 == 0) & (pos_prior == 0),                 # Zero-demand only across entire dataset
        (pos_2025 == 0) & (pos_prior > 0),                  # Inactive / Discontinued in 2025
        (pos_2025 > 0) & (pos_prior == 0) & (pos_2025 < 3), # TRUE Data Insufficiency: 2025-only with <3 records
        (pos_2025 > 0) & (pos_prior == 0) & (pos_2025 >= 3),# 2025 New Launch with >=3 records
        (pos_2025 > 0) & (pos_prior > 0) & (pos_2025 < 3),  # Multi-Year Active (1-2 recs in 2025 -> NOT insufficient!)
    ]
    labels = [
        "1. Zero-Demand Only (Never Active)",
        "2. Inactive in 2025 (No 2025 Demand)",
        "3. Insufficient History (2025-Only, <3 Pos Records)",
        "4. 2025 New Launch (>=3 Pos Records)",
        "5. Multi-Year Active (2023/24+25; 1-2 Recs in 25 - Sufficient)",
    ]
    m["Sufficiency_Status"] = np.select(conds, labels, default="6. Multi-Year Active (2023/24+25; >=3 Recs in 25 - Sufficient)")
    return m

bulk_level_df = classify_bulk_sufficiency(df_clean, ["Bulk Value"])
bulk_reg_df = classify_bulk_sufficiency(df_clean, ["Bulk Value", "Region Id"])

df_s10_bulk = (
    bulk_level_df.groupby("Sufficiency_Status", as_index=False)
    .agg(Bulk_Count=("Bulk Value", "count"), Total_Quantity=("Total_Qty", "sum"))
    .assign(
        Bulk_Share_Pct=lambda d: (d["Bulk_Count"] / len(bulk_level_df) * 100).round(2),
        Volume_Share_Pct=lambda d: (d["Total_Quantity"] / bulk_level_df["Total_Qty"].sum() * 100).round(2),
    )
    .sort_values("Sufficiency_Status")
    .reset_index(drop=True)
)

df_s10_bulk_reg = (
    bulk_reg_df.groupby("Sufficiency_Status", as_index=False)
    .agg(Bulk_Region_Count=("Bulk Value", "count"), Total_Quantity=("Total_Qty", "sum"))
    .assign(
        Series_Share_Pct=lambda d: (d["Bulk_Region_Count"] / len(bulk_reg_df) * 100).round(2),
        Volume_Share_Pct=lambda d: (d["Total_Quantity"] / bulk_reg_df["Total_Qty"].sum() * 100).round(2),
    )
    .sort_values("Sufficiency_Status")
    .reset_index(drop=True)
)

# Inactive Part by Business Segment (Active = positive demand in 2025; Inactive = no positive demand in 2025)
part_seg_2025 = (
    df_clean.groupby(["Business Segment Description", "Part Name"], as_index=False)
    .agg(Qty_2025=("Quantity", lambda s: s[df_clean.loc[s.index, "Year"] == 2025].sum()))
)
part_seg_2025["Is_Active_2025"] = part_seg_2025["Qty_2025"] > 0

df_s10_inactive_seg = (
    part_seg_2025.groupby("Business Segment Description", as_index=False)
    .agg(
        Total_Distinct_Parts=("Part Name", "count"),
        Active_Parts_2025=("Is_Active_2025", "sum"),
        Inactive_Parts_2025=("Is_Active_2025", lambda s: (~s).sum()),
    )
    .assign(Inactive_Part_Pct=lambda d: (d["Inactive_Parts_2025"] / d["Total_Distinct_Parts"] * 100).round(2))
    .sort_values("Inactive_Parts_2025", ascending=False)
    .reset_index(drop=True)
)

print("--- Slide 10A: Bulk Value Coverage & Sufficiency ---")
display(df_s10_bulk)
print("--- Slide 10B: Inactive Parts by Business Segment ---")
display(df_s10_inactive_seg)

# Two-panel Slide 10 chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15.5, 5.5))

# Left: Bulk Value Sufficiency Breakdown
b_plot = df_s10_bulk.iloc[::-1].reset_index(drop=True)
short_labels = [
    s.replace("1. ", "").replace("2. ", "").replace("3. ", "").replace("4. ", "").replace("5. ", "").replace("6. ", "")
    for s in b_plot["Sufficiency_Status"]
]
s10_colors = [TEAL, BLUE, "#48CAE4", ORANGE, SLATE, "#ADB5BD"][:len(b_plot)][::-1]
bars1 = ax1.barh(short_labels, b_plot["Bulk_Count"], color=s10_colors, height=0.58)
ax1.set_title("Bulk Value Portfolio Coverage & Sufficiency", fontsize=11.5, fontweight="bold", color=NAVY, pad=10)
ax1.set_xlabel("Distinct Bulk Values", fontsize=9.5)
ax1.set_xlim(0, b_plot["Bulk_Count"].max() * 1.28)
ax1.grid(axis="x", linestyle="--", alpha=0.4)
ax1.grid(axis="y", visible=False)
for bar, (_, r) in zip(bars1, b_plot.iterrows()):
    ax1.text(bar.get_width(), bar.get_y() + bar.get_height() / 2,
             f" {int(r['Bulk_Count']):,} ({r['Bulk_Share_Pct']:.1f}%)",
             va="center", ha="left", fontsize=8.5, fontweight="bold")

# Right: Active vs Inactive Parts by Business Segment (Stacked Horizontal Bar)
seg_inact = df_s10_inactive_seg.iloc[::-1].reset_index(drop=True)
ys2 = np.arange(len(seg_inact))
ax2.barh(ys2, seg_inact["Active_Parts_2025"], color=TEAL, height=0.58, label="Active Parts (2025 Demand > 0)")
ax2.barh(ys2, seg_inact["Inactive_Parts_2025"], left=seg_inact["Active_Parts_2025"], color=ORANGE, height=0.58, label="Inactive Parts (0 Demand in 2025)")
ax2.set_yticks(ys2)
ax2.set_yticklabels(seg_inact["Business Segment Description"], fontsize=9)
ax2.set_xlim(0, seg_inact["Total_Distinct_Parts"].max() * 1.22)
ax2.set_title("Active vs. Inactive Parts by Business Segment", fontsize=11.5, fontweight="bold", color=NAVY, pad=10)
ax2.set_xlabel("Distinct Part Count", fontsize=9.5)
ax2.legend(loc="lower right", frameon=False, fontsize=8.5)
ax2.grid(axis="x", linestyle="--", alpha=0.4)
ax2.grid(axis="y", visible=False)

for idx_s, r in seg_inact.iterrows():
    ax2.text(r["Total_Distinct_Parts"] + seg_inact["Total_Distinct_Parts"].max() * 0.015, idx_s,
             f"{int(r['Inactive_Parts_2025']):,} inactive ({r['Inactive_Part_Pct']:.1f}%)",
             va="center", ha="left", fontsize=8.5, color=NAVY, fontweight="bold")

sns.despine(top=True, right=True)
plt.tight_layout()
s10_png = os.path.join(OUT_DIR, "Slide10_Bulk_Sufficiency_and_Inactive_Segment.png")
plt.savefig(s10_png, dpi=300)
plt.show()

## Cell 8: Slide 11 — Outliers & Quantity Anomalies (Zero Bar Removed & `Bulk × Region` Frequency)

In [ ]:
# ==========================================
# SLIDE 11: OUTLIERS & QUANTITY ANOMALIES
# ==========================================
pos_tx = df_clean[df_clean["Quantity"] > 0].copy()
q1_g, q3_g = pos_tx["Quantity"].quantile(0.25), pos_tx["Quantity"].quantile(0.75)
iqr_g = q3_g - q1_g
fence_global_15 = q3_g + 1.5 * iqr_g
fence_global_30 = q3_g + 3.0 * iqr_g
p99_g = pos_tx["Quantity"].quantile(0.99)

def per_group_iqr_mask(df, keys, min_obs=MIN_OBS_IQR):
    g = df.loc[df["Quantity"] > 0].groupby(keys)["Quantity"]
    stats = pd.concat([g.size().rename("n"), g.quantile(0.25).rename("q1"), g.quantile(0.75).rename("q3")], axis=1).reset_index()
    stats["fence"] = np.where(stats["n"] >= min_obs, stats["q3"] + 1.5 * (stats["q3"] - stats["q1"]), np.nan)
    m = df[keys + ["Quantity"]].merge(stats[keys + ["fence"]], on=keys, how="left")
    return m["Quantity"].values > m["fence"].values

df_clean["Outlier_Global_IQR"] = df_clean["Quantity"] > fence_global_15
df_clean["Outlier_Global_Extreme"] = df_clean["Quantity"] > fence_global_30
df_clean["Outlier_P99"] = df_clean["Quantity"] > p99_g
df_clean["Outlier_PerSeries_PartReg"] = per_group_iqr_mask(df_clean, ["Part Name", "Region Id"])
df_clean["Outlier_PerSeries_BulkReg"] = per_group_iqr_mask(df_clean, ["Bulk Value", "Region Id"])

# Classification comparison table (NO zero bar!)
n_clean = len(df_clean)
n_pos_clean = len(pos_tx)
s11_classes = [
    ("Global IQR (> Q3 + 1.5×IQR)", f"Pooled threshold > {fence_global_15:,.0f} across all products", int(df_clean["Outlier_Global_IQR"].sum())),
    ("Global Extreme (> Q3 + 3.0×IQR)", f"Pooled extreme threshold > {fence_global_30:,.0f}", int(df_clean["Outlier_Global_Extreme"].sum())),
    ("Global Top 1% (> P99)", f"Top 1% of positive shipments (> {p99_g:,.0f})", int(df_clean["Outlier_P99"].sum())),
    ("Per-Series IQR (Bulk × Region)", f"Within each Bulk+Region (> Q3 + 1.5×IQR, n>={MIN_OBS_IQR})", int(df_clean["Outlier_PerSeries_BulkReg"].sum())),
    ("Per-Series IQR (Part × Region)", f"Within each Part+Region (> Q3 + 1.5×IQR, n>={MIN_OBS_IQR})", int(df_clean["Outlier_PerSeries_PartReg"].sum())),
]
df_s11_classification = pd.DataFrame([
    {
        "Outlier Classification": name,
        "Rule / Definition": rule,
        "Flagged Records": cnt,
        "% of Clean Rows": round(cnt / n_clean * 100, 2),
        "% of Positive Rows": round(cnt / n_pos_clean * 100, 2),
    }
    for name, rule, cnt in s11_classes
])

# Outlier Frequency at Bulk Value x Region Id level
df_s11_bulk_reg = (
    df_clean.groupby(["Bulk Value", "Region Id"], as_index=False)
    .agg(
        Total_Records=("Quantity", "size"),
        Positive_Records=("Quantity", lambda s: int((s > 0).sum())),
        Outlier_Records=("Outlier_PerSeries_BulkReg", "sum"),
        Max_Quantity=("Quantity", "max"),
        Median_Pos_Qty=("Quantity", lambda s: s[s > 0].median() if (s > 0).any() else 0.0),
    )
    .assign(
        Outlier_Freq_All_Pct=lambda d: (d["Outlier_Records"] / d["Total_Records"] * 100).round(2),
        Outlier_Freq_Pos_Pct=lambda d: np.where(d["Positive_Records"] > 0, (d["Outlier_Records"] / d["Positive_Records"] * 100).round(2), 0.0),
    )
)

# Frequency distribution bucket across Bulk x Region series
freq_bins = [-0.01, 0, 2, 5, 10, 100]
freq_labels = ["0% (No Outliers)", "0.1% – 2% Rare", "2.1% – 5% Occasional", "5.1% – 10% Moderate", "> 10% Frequent"]
df_s11_bulk_reg["Frequency_Bucket"] = pd.cut(df_s11_bulk_reg["Outlier_Freq_Pos_Pct"], bins=freq_bins, labels=freq_labels)

df_s11_freq_dist = (
    df_s11_bulk_reg.groupby("Frequency_Bucket", observed=False, as_index=False)
    .agg(Bulk_Region_Series=("Bulk Value", "count"), Total_Outlier_Records=("Outlier_Records", "sum"))
    .assign(Series_Share_Pct=lambda d: (d["Bulk_Region_Series"] / len(df_s11_bulk_reg) * 100).round(2))
)

df_s11_top_bulk_reg = (
    df_s11_bulk_reg[df_s11_bulk_reg["Outlier_Records"] > 0]
    .sort_values(["Outlier_Records", "Outlier_Freq_Pos_Pct"], ascending=False)
    .head(20)
    .reset_index(drop=True)
)

print("--- Slide 11A: Outlier Classification Comparison (Zero Bar Removed) ---")
display(df_s11_classification)
print("--- Slide 11B: Outlier Frequency Distribution at Bulk × Region Level ---")
display(df_s11_freq_dist)
print("--- Slide 11C: Top 20 Bulk × Region Series by Outlier Count ---")
display(df_s11_top_bulk_reg)

# Two-panel Slide 11 Chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15.5, 5.2))

# Left: Outlier Classification Bar Chart (NO zero bar!)
c_plot = df_s11_classification.iloc[::-1].reset_index(drop=True)
bars_c = ax1.barh(c_plot["Outlier Classification"], c_plot["Flagged Records"],
                  color=[NAVY, TEAL, SLATE, BLUE, AMBER], height=0.55)
ax1.set_title("Anomaly Count by Outlier Classification (Zero Records Excluded)", fontsize=11.5, fontweight="bold", color=NAVY, pad=10)
ax1.set_xlabel("Flagged Transaction Records", fontsize=9.5)
ax1.set_xlim(0, c_plot["Flagged Records"].max() * 1.28)
ax1.xaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
ax1.grid(axis="x", linestyle="--", alpha=0.4)
ax1.grid(axis="y", visible=False)
for bar, (_, r) in zip(bars_c, c_plot.iterrows()):
    ax1.text(bar.get_width(), bar.get_y() + bar.get_height() / 2,
             f" {int(r['Flagged Records']):,} ({r['% of Positive Rows']:.1f}% pos)",
             va="center", ha="left", fontsize=8.5, fontweight="bold")

# Right: Bulk x Region Outlier Frequency Distribution
f_plot = df_s11_freq_dist
bars_f = ax2.bar(f_plot["Frequency_Bucket"].astype(str), f_plot["Bulk_Region_Series"],
                 color=[TEAL, "#48CAE4", AMBER, ORANGE, "#D62828"], width=0.55)
ax2.set_title("Outlier Frequency Distribution Across Bulk × Region Series", fontsize=11.5, fontweight="bold", color=NAVY, pad=10)
ax2.set_ylabel("Number of Bulk × Region Series", fontsize=9.5)
ax2.tick_params(axis="x", rotation=20)
ax2.set_ylim(0, f_plot["Bulk_Region_Series"].max() * 1.18)
ax2.grid(axis="y", linestyle="--", alpha=0.4)
ax2.grid(axis="x", visible=False)
for bar, (_, r) in zip(bars_f, f_plot.iterrows()):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
             f"{int(r['Bulk_Region_Series']):,}\n({r['Series_Share_Pct']:.1f}%)",
             va="bottom", ha="center", fontsize=8.5, fontweight="bold")

sns.despine(top=True, right=True)
plt.tight_layout()
s11_png = os.path.join(OUT_DIR, "Slide11_Outliers_and_BulkRegion_Freq.png")
plt.savefig(s11_png, dpi=300)
plt.show()

## Cell 9: Export All Slide Tables to Excel (`ClariCast_Slide_Tables.xlsx`)

In [ ]:
# ==========================================
# EXPORT ALL SLIDE TABLES TO EXCEL
# ==========================================
slide_sheets = {
    "Slide5_Missing_Values": df_s5_missing,
    "Slide6_Annual_Summary": df_s6_annual,
    "Slide6_Monthly_Trend": monthly_chem[["YM_Str", "Year", "Total_Quantity", "Active_Parts", "Active_Bulks", "Records"]],
    "Slide7_Data_Grain": df_s7_grain,
    "Slide8_Hierarchy_Map": df_s8_mapping,
    "Slide9_Region_Share": df_s9_region,
    "Slide9_Segment_Share": df_s9_segment,
    "Slide10_Bulk_Sufficiency": df_s10_bulk,
    "Slide10_BulkReg_Suffic": df_s10_bulk_reg,
    "Slide10_Inactive_By_Seg": df_s10_inactive_seg,
    "Slide11_Outlier_Classes": df_s11_classification,
    "Slide11_BulkReg_FreqDist": df_s11_freq_dist,
    "Slide11_Top_BulkReg_Outl": df_s11_top_bulk_reg,
}

with pd.ExcelWriter(EXCEL_OUT, engine="openpyxl") as writer:
    for sheet_name, tbl in slide_sheets.items():
        tbl.to_excel(writer, sheet_name=sheet_name[:31], index=False)

print(f"Exported all {len(slide_sheets)} slide tables -> '{EXCEL_OUT}'")
print(f"All 7 slide charts saved in -> '{OUT_DIR}/'")